# Retail Performance & Planning Lab

## Data Overview

This project uses the local Superstore sample dataset for an Analyst job application portfolio. This notebook reviews the dataset structure, date coverage, record grain, and basic data quality.

### Key Findings

- `Orders`: 10,194 sales line items and 21 fields, covering 5,111 distinct orders.
- `People`: 4 regional manager mappings and 2 fields. `Returns`: 296 order-level return flags and 2 fields.
- Order dates: **2023-01-03 to 2026-12-30**. Ship dates: **2023-01-07 to 2027-01-05**.
- No missing values, blank strings, or exact duplicate rows were detected in any sheet. Excluding `Row ID`, `Orders` contains 2 additional duplicate records, retained for review.
- The source has an `.xlsx` extension but is a legacy Excel binary workbook. It was successfully read using `xlrd`.

The source workbook is read-only. All narrative, code, variable names, and output labels use English.

## Scope and Assumptions

The only data source is `data/raw/sample_-_superstore.xlsx`. The original workbook is not converted, overwritten, or cleaned.

Dates are interpreted as calendar dates in the workbook without timezone conversion. `Sales` and `Profit` are provisionally interpreted as line-item amounts. Currency, tax treatment, and the cost basis for profit are not confirmed by a local data dictionary. Amounts are therefore not labeled as USD, and net revenue is not inferred.

**2026-11-01** is the inspection reference date. It is used only to identify later records, not to filter the dataset. The sample includes future dates and should not be described as actual realized business performance as of that reference date.

## 1. Project and Python Environment

Select the project's `.venv` kernel in the IDE and run the notebook from top to bottom. Path discovery supports either the project root or `notebooks/` as the working directory.

In [1]:
from pathlib import Path
from hashlib import sha256
from importlib.metadata import version
import sys
import pandas as pd
from IPython.display import display

project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data/raw/sample_-_superstore.xlsx").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project root or its notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from retail_lab.analysis import (
    FIELD_MEANINGS, analyze_workbook, detect_excel_engine, load_workbook,
)

source_path = project_root / "data/raw/sample_-_superstore.xlsx"
source_hash_before = sha256(source_path.read_bytes()).hexdigest()
as_of_date = pd.Timestamp("2026-11-01")

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Virtual environment: {sys.prefix != sys.base_prefix}")
assert Path(sys.prefix).resolve() == (project_root / ".venv").resolve(), "Select the project .venv kernel."
display(pd.DataFrame([
    {"Package": package, "Version": version(package)}
    for package in ["pandas", "openpyxl", "xlrd", "ipykernel", "nbformat", "nbclient", "nbconvert"]
]))

Python executable: C:\Users\Larry\retail-sales-performance-analysis\.venv\Scripts\python.exe
Python version: 3.14.6
Virtual environment: True


,Package,Version
0,pandas,3.0.5
1,openpyxl,3.1.5
2,xlrd,2.0.2
3,ipykernel,7.3.0
4,nbformat,5.10.4
5,nbclient,0.11.0
6,nbconvert,7.17.1


In [2]:
project_entries = []
for entry_name in ["data/raw", "notebooks", "sql", "dashboard", "images"]:
    entry_path = project_root / entry_name
    project_entries.append({
        "Path": entry_name,
        "Exists": entry_path.exists(),
        "File count": sum(path.is_file() for path in entry_path.rglob("*")),
    })
display(pd.DataFrame(project_entries))

,Path,Exists,File count
0,data/raw,True,1
1,notebooks,True,1
2,sql,True,1
3,dashboard,True,1
4,images,True,1


## 2. Load All Worksheets Without Modifying the Source

Inspect the file signature before choosing the reader rather than relying on the extension alone. `sheet_name=None` loads every worksheet. All reported row counts exclude headers.

In [3]:
excel_engine = detect_excel_engine(source_path)
sheets = load_workbook(source_path)
phase1_metrics, phase1_tables = analyze_workbook(sheets, as_of_date)
orders = sheets["Orders"]
people = sheets["People"]
returns = sheets["Returns"]
print(f"Detected engine: {excel_engine}")
print(f"Source size (bytes): {source_path.stat().st_size:,}")
display(pd.DataFrame([
    {"Sheet": sheet_name, "Data rows": len(frame), "Columns": len(frame.columns)}
    for sheet_name, frame in sheets.items()
]))

Detected engine: xlrd
Source size (bytes): 3,422,208


,Sheet,Data rows,Columns
0,Orders,10194,21
1,People,4,2
2,Returns,296,2


### Line-Item Previews

Show the first and last records and a sample with a fixed random seed without changing the source data. One row can contain multiple units of the same product, so the number of line items is not the number of units sold.

In [4]:
preview_columns = ["Row ID", "Order ID", "Order Date", "Product ID", "Quantity", "Sales", "Profit"]
orders[preview_columns].head()

,Row ID,Order ID,Order Date,Product ID,Quantity,Sales,Profit
0,1,US-2023-103800,2023-01-03,OFF-PA-10000174,2,16.448,5.5512
1,2,US-2023-112326,2023-01-04,OFF-BI-10004094,2,3.540,-5.4870
2,3,US-2023-112326,2023-01-04,OFF-LA-10003223,3,11.784,4.2717
3,4,US-2023-112326,2023-01-04,OFF-ST-10002743,3,272.736,-64.7748
4,5,US-2023-141817,2023-01-05,OFF-AR-10003478,3,19.536,4.8840


In [5]:
orders[preview_columns].tail()

,Row ID,Order ID,Order Date,Product ID,Quantity,Sales,Profit
10189,10190,US-2026-143259,2026-12-30,OFF-BI-10003684,3,52.776,19.7910
10190,10191,US-2026-115427,2026-12-30,OFF-BI-10004632,2,20.720,6.4750
10191,10192,US-2026-156720,2026-12-30,OFF-FA-10003472,3,3.024,-0.6048
10192,10193,US-2026-143259,2026-12-30,TEC-PH-10004774,7,90.930,2.7279
10193,10194,CA-2026-143500,2026-12-30,OFF-BI-10004040,3,3.024,-0.6048


In [6]:
orders[preview_columns].sample(5, random_state=42)

,Row ID,Order ID,Order Date,Product ID,Quantity,Sales,Profit
6636,6637,US-2025-108777,2025-12-12,OFF-BI-10003982,3,18.693,-14.3313
3742,3743,US-2024-102778,2024-11-21,FUR-FU-10000087,1,18.176,4.7712
5114,5115,US-2025-153577,2025-06-27,FUR-CH-10003981,3,539.658,-7.7094
6505,6506,US-2025-161025,2025-12-03,OFF-ST-10001932,4,772.680,108.1752
10072,10073,US-2026-155376,2026-12-22,OFF-AP-10001058,3,839.430,218.2518


## 3. Fields and Read Data Types

List every field and its meaning. `Postal Code` contains both numeric US ZIP codes and Canadian text codes. Treat it as a geographic identifier rather than a numeric measure. Values are preserved as read in this inspection. Descriptions are based on field names and sample records; detailed financial definitions remain unconfirmed.

In [7]:
field_meanings = FIELD_MEANINGS
field_catalog = pd.DataFrame([
    {"Sheet": sheet_name, "Field": column,
     "Read dtype": str(frame[column].dtype),
     "Meaning": field_meanings[column]}
    for sheet_name, frame in sheets.items() for column in frame.columns
])
with pd.option_context("display.max_rows", 40, "display.max_colwidth", 60):
    display(field_catalog)

,Sheet,Field,Read dtype,Meaning
0,Orders,Row ID,int64,Source line-item identifier
1,Orders,Order ID,str,Order identifier; one order can contain multiple rows
2,Orders,Order Date,datetime64[us],Date the order was placed
3,Orders,Ship Date,datetime64[us],"Shipment date, not delivery date"
4,Orders,Ship Mode,str,Shipping service class
5,Orders,Customer ID,str,Customer identifier
6,Orders,Customer Name,str,Customer name
7,Orders,Segment,str,Customer segment
8,Orders,Country/Region,str,Country or geographic jurisdiction
9,Orders,City,str,City


## 4. Date Coverage

`People` and `Returns` have no date fields. These sheets cannot establish manager tenure or the date on which a return occurred.

In [8]:
date_records = []
for sheet_name, frame in sheets.items():
    for column in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[column]):
            date_records.append({
                "Sheet": sheet_name, "Date field": column,
                "Earliest date": frame[column].min().date().isoformat(),
                "Latest date": frame[column].max().date().isoformat(),
                "Missing dates": int(frame[column].isna().sum()),
            })
display(pd.DataFrame(date_records))
print(f"Order lines after {as_of_date.date()}: {(orders['Order Date'] > as_of_date).sum():,}")

,Sheet,Date field,Earliest date,Latest date,Missing dates
0,Orders,Order Date,2023-01-03,2026-12-30,0
1,Orders,Ship Date,2023-01-07,2027-01-05,0


Order lines after 2026-11-01: 938


## 5. What Each Row Represents

- **Orders**: one product sales line item within an order. `Row ID` is unique in this file. Count orders using distinct `Order ID` values and aggregate sales, units, and profit from the line-item measures. The 10,194 rows represent line items, not 10,194 distinct orders.
- **People**: one sales region and its regional manager. `Region` is unique and can serve as the join key for a regional dimension table.
- **Returns**: one order-level return flag. `Order ID` is unique, and all observed flags are `Yes`. Product identifiers, returned quantities, return amounts, and return dates are absent, so the sheet cannot determine which line items were returned, how many units were returned, or the financial amount of returns.

In [9]:
order_line_counts = orders.groupby("Order ID").size()
grain_summary = pd.DataFrame([
    {"Check": "Order lines", "Value": len(orders)},
    {"Check": "Unique Row ID", "Value": orders["Row ID"].nunique()},
    {"Check": "Distinct orders", "Value": orders["Order ID"].nunique()},
    {"Check": "Orders with multiple lines", "Value": int(order_line_counts.gt(1).sum())},
    {"Check": "Extra rows sharing Order ID and Product ID", "Value": int(orders.duplicated(["Order ID", "Product ID"]).sum())},
    {"Check": "Unique People regions", "Value": people["Region"].nunique()},
    {"Check": "Unique returned Order ID", "Value": returns["Order ID"].nunique()},
])
display(grain_summary)
example_order_id = orders.loc[1, "Order ID"]
display(orders.loc[orders["Order ID"].eq(example_order_id), preview_columns])

,Check,Value
0,Order lines,10194
1,Unique Row ID,10194
2,Distinct orders,5111
3,Orders with multiple lines,2518
4,Extra rows sharing Order ID and Product ID,11
5,Unique People regions,4
6,Unique returned Order ID,296


,Row ID,Order ID,Order Date,Product ID,Quantity,Sales,Profit
1,2,US-2023-112326,2023-01-04,OFF-BI-10004094,2,3.540,-5.4870
2,3,US-2023-112326,2023-01-04,OFF-LA-10003223,3,11.784,4.2717
3,4,US-2023-112326,2023-01-04,OFF-ST-10002743,3,272.736,-64.7748


There are 11 additional rows sharing an `Order ID + Product ID` combination. This combination is not a unique key. Repeated products within an order may reflect split line items or source duplication. Review them before deciding whether any record should be removed.

## 6. Basic Data Quality Checks

Check missing values, blank strings, exact duplicates, duplicates after excluding the row identifier, date ordering, and join-key coverage. Negative profit can represent a loss and is not automatically a data error.

In [10]:
phase1_tables["sheet_summary"][[
    "sheet", "data_rows", "columns", "null_cells",
    "blank_string_cells", "exact_duplicate_extra_rows",
]]

,sheet,data_rows,columns,null_cells,blank_string_cells,exact_duplicate_extra_rows
0,Orders,10194,21,0,0,0
1,People,4,2,0,0,0
2,Returns,296,2,0,0,0


In [11]:
quality_checks = phase1_tables["quality_checks"]
display(quality_checks.loc[quality_checks["count"].gt(0), [
    "sheet", "field", "check", "count", "row_denominator", "rate",
]])
display(phase1_tables["duplicate_summary"][[
    "sheet", "duplicate_type", "extra_rows", "affected_rows", "missing_key_rows",
]])

,sheet,field,check,count,row_denominator,rate
5,Orders,Order Date,after_reference_date,938,10194,0.092015
8,Orders,Ship Date,after_reference_date,969,10194,0.095056
24,Orders,Sales,iqr_outlier,1183,10194,0.116049
31,Orders,Profit,iqr_outlier,1913,10194,0.187659
34,Orders,Profit,negative_value,1901,10194,0.186482


,sheet,duplicate_type,extra_rows,affected_rows,missing_key_rows
0,Orders,full_row,0,0,0
1,Orders,order_id,5083,7601,0
2,Orders,order_line_id,0,0,0
3,Orders,content_excluding_row_id,2,4,0
4,Orders,order_product,11,22,0
5,People,full_row,0,0,0
6,People,region_key,0,0,0
7,Returns,full_row,0,0,0
8,Returns,order_id,0,0,0


### Duplicate Content Requiring Review

Excluding `Row ID`, there are 2 additional exact duplicate records involving 4 rows in total. This indicates potential duplication but does not establish that records should be deleted. Display identifying fields for review and retain every record.

In [12]:
duplicate_content_mask = orders.drop(columns="Row ID").duplicated(keep=False)
orders.loc[duplicate_content_mask, preview_columns]

,Row ID,Order ID,Order Date,Product ID,Quantity,Sales,Profit
390,391,US-2023-150119,2023-04-23,FUR-CH-10002965,2,281.372,-12.0588
391,392,US-2023-150119,2023-04-23,FUR-CH-10002965,2,281.372,-12.0588
1698,1699,CA-2023-153623,2023-11-24,FUR-FU-10002501,8,99.120,35.4144
1699,1700,CA-2023-153623,2023-11-24,FUR-FU-10002501,8,99.120,35.4144


### Numeric Measure Summary

Summarize business measures only. Row identifiers and postal codes are not business measures.

In [13]:
orders[["Sales", "Quantity", "Discount", "Profit"]].describe().round(4)

,Sales,Quantity,Discount,Profit
count,10194.0000,10194.0000,10194.0000,10194.0000
mean,228.2259,3.7918,0.1554,28.6734
std,619.9068,2.2283,0.2062,232.4651
min,0.4440,1.0000,0.0000,-6599.9780
25%,17.2200,2.0000,0.0000,1.7608
50%,53.9100,3.0000,0.2000,8.6900
75%,209.5000,5.0000,0.2000,29.2979
max,22638.4800,14.0000,0.8000,8399.9760


## 7. Implications for Further Analysis

1. Count orders using distinct `Order ID` values. Aggregate product sales at the line-item level and retain `Row ID` for traceability.
2. The current right-side keys are unique when joining `People` on `Region` or `Returns` on `Order ID`. Use `validate="many_to_one"` in actual joins and reconcile row counts before and after each join.
3. `Returns` supports a flag indicating whether an order appears in the return list. Absence from that list does not establish that no return occurred unless the list is known to be complete.
4. Retain the 2 additional duplicate records identified after excluding the row identifier for further review. Retain all 1,901 negative-profit line items for profitability analysis.
5. The sample covers the United States and Canada, with postal codes stored as a mixture of numbers and text. Standardize identifier formats in a derived copy and confirm the currency treatment of cross-country amounts before financial comparisons.
6. There are 938 line items with order dates after the reference date of 2026-11-01. Present the project as a sample-data analysis. Any separate as-of analysis should apply an explicit date filter and state its coverage.

This review covers structure and basic data quality. It does not verify the business accuracy of every field or establish how the sample was originally generated.

## 8. Source File Integrity

Compare the SHA-256 hash before and after execution to confirm that the source file's bytes have not changed.

In [14]:
source_hash_after = sha256(source_path.read_bytes()).hexdigest()
assert source_hash_after == source_hash_before, "Raw source file changed during inspection."
print("Raw source unchanged:", source_hash_before == source_hash_after)
print("SHA-256:", source_hash_after)

Raw source unchanged: True
SHA-256: 5045ed4fec49cae2cc0dfc5ab0d204ba09efa767fe00c996822499e8cc42fbbd


## 9. Phase 1 Base Metrics and Reconciliation

The shared `retail_lab.analysis` module reuses the loading and field definitions established in this overview. The command-line entry point is `scripts/run_analysis.py`; it writes the full quality profile, base metrics, dimensional summaries, and validation results into `outputs/`.

All metrics below are computed directly from Orders without enrichment joins. Profit Margin is total Profit divided by total Sales, not an average of line-item margins. Missing or incomplete measures and zero Sales produce an unavailable margin. No source rows are removed.

In [15]:
pd.DataFrame([phase1_metrics]).T.rename(columns={0: "Value"})

,Value
sales,2326534.3543
profit,292296.8146
profit_margin,0.125636
order_count,5111
order_line_count,10194
sales_status,complete
profit_status,complete
profit_margin_status,available
sales_unavailable_line_count,0
profit_unavailable_line_count,0


### Coverage and Reconciliation

Monthly, regional, and category summaries retain missing grouping values and invalid order dates in an unassigned group. Sales, Profit, and line counts reconcile to grand totals. Order Count uses the distinct union across groups because an order may span multiple categories. Group order-count sums are informational only.

The current file has 48 observed months, 4 regions, and 3 categories. All 12 required reconciliation checks pass; 3 additional rows report group-order-count sums for information.

In [16]:
display(phase1_tables["reconciliation"][[
    "dimension", "check", "expected", "observed", "difference", "status",
]])

,dimension,check,expected,observed,difference,status
0,Month,sales,2.326534e+06,2.326534e+06,0.000000e+00,PASS
1,Month,profit,2.922968e+05,2.922968e+05,0.000000e+00,PASS
2,Month,order_line_count,1.019400e+04,1.019400e+04,0.000000e+00,PASS
3,Month,order_count_union,5.111000e+03,5.111000e+03,0.000000e+00,PASS
4,Month,order_count_sum_information,5.111000e+03,5.111000e+03,0.000000e+00,INFO
5,Region,sales,2.326534e+06,2.326534e+06,0.000000e+00,PASS
6,Region,profit,2.922968e+05,2.922968e+05,0.000000e+00,PASS
7,Region,order_line_count,1.019400e+04,1.019400e+04,0.000000e+00,PASS
8,Region,order_count_union,5.111000e+03,5.111000e+03,0.000000e+00,PASS
9,Region,order_count_sum_information,5.111000e+03,5.111000e+03,0.000000e+00,INFO


### Worksheet Relationship Checks

People maps Region to a regional manager. Returns adds order-level return flags, not returned-unit counts or financial adjustments. Trial left joins validate many-to-one cardinality and preserve line counts and amounts. They are not used as the source of the metric calculations.

In [17]:
phase1_tables["relationships"][[
    "right_sheet", "key", "status", "joined_row_count",
    "row_count_preserved", "sales_preserved", "profit_preserved",
]]

,right_sheet,key,status,joined_row_count,row_count_preserved,sales_preserved,profit_preserved
0,People,Region,PASS,10194,True,True,True
1,Returns,Order ID,PASS,10194,True,True,True
